[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/volume2_chapter_02/notebook_2B_byzantine_fl.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 2B: Byzantine-Robust Federated Learning

**Volume 2, Chapter 2: Security, Privacy, and Robustness**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement a basic federated learning simulation with multiple participants
2. Simulate Byzantine attacks including random noise and targeted poisoning
3. Compare aggregation methods: FedAvg, coordinate-wise median, trimmed mean, and Krum
4. Evaluate trade-offs between Byzantine tolerance and convergence speed
5. Design appropriate defenses for healthcare federated learning consortia

## Clinical Context

Federated learning enables hospitals to collaboratively train AI models without sharing patient data. However, this distributed approach creates security vulnerabilities:

- A compromised hospital can send malicious model updates
- Software bugs can cause unintentionally harmful updates
- Network issues can corrupt updates in transit

Byzantine-robust aggregation methods protect against these threats by limiting the influence of outlier updates.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from typing import List, Dict, Callable
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'fedavg': '#3498DB',
    'median': '#27AE60',
    'trimmed': '#9B59B6',
    'krum': '#E74C3C',
    'attack': '#F39C12',
    'honest': '#1ABC9C'
}

print("✓ Libraries imported successfully")

---

## 1. Simulated Healthcare Federated Learning Setup

We simulate a consortium of hospitals collaboratively training a sepsis prediction model.

In [ ]:
def generate_hospital_data(n_hospitals=5, samples_per_hospital=500, n_features=10, seed=42):
    """
    Generate synthetic healthcare data distributed across hospitals.
    Each hospital has slightly different patient distributions.
    """
    np.random.seed(seed)
    
    hospital_data = []
    
    for h in range(n_hospitals):
        # Base classification data
        X, y = make_classification(
            n_samples=samples_per_hospital,
            n_features=n_features,
            n_informative=7,
            n_redundant=2,
            n_clusters_per_class=2,
            flip_y=0.05,
            random_state=seed + h
        )
        
        # Add hospital-specific shift (heterogeneous data)
        hospital_shift = np.random.randn(n_features) * 0.3
        X = X + hospital_shift
        
        hospital_data.append({
            'hospital_id': h,
            'X': X,
            'y': y,
            'n_samples': len(y),
            'prevalence': y.mean()
        })
    
    return hospital_data

# Generate federated dataset
N_HOSPITALS = 10
N_FEATURES = 10

hospital_data = generate_hospital_data(
    n_hospitals=N_HOSPITALS,
    samples_per_hospital=400,
    n_features=N_FEATURES
)

# Create global test set
X_test_global, y_test_global = make_classification(
    n_samples=1000,
    n_features=N_FEATURES,
    n_informative=7,
    n_redundant=2,
    n_clusters_per_class=2,
    flip_y=0.05,
    random_state=999
)

print(f"Federated Learning Setup:")
print(f"  Number of hospitals: {N_HOSPITALS}")
print(f"  Features per sample: {N_FEATURES}")
print(f"  Global test set size: {len(y_test_global)}")
print(f"\nHospital data distribution:")
for h in hospital_data:
    print(f"  Hospital {h['hospital_id']}: {h['n_samples']} samples, {h['prevalence']:.1%} positive")

### Simple Logistic Regression Implementation

We implement a simple logistic regression that exposes gradients for federated learning.

In [ ]:
class SimpleLogisticRegression:
    """
    Simple logistic regression for federated learning demonstration.
    Exposes weights and gradients for aggregation.
    """
    
    def __init__(self, n_features, learning_rate=0.1):
        self.n_features = n_features
        self.lr = learning_rate
        # Initialize weights (including bias as last element)
        self.weights = np.zeros(n_features + 1)
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def predict_proba(self, X):
        X_bias = np.column_stack([X, np.ones(len(X))])
        return self.sigmoid(X_bias @ self.weights)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)
    
    def compute_gradient(self, X, y):
        """Compute gradient of cross-entropy loss."""
        X_bias = np.column_stack([X, np.ones(len(X))])
        predictions = self.sigmoid(X_bias @ self.weights)
        error = predictions - y
        gradient = (X_bias.T @ error) / len(y)
        return gradient
    
    def train_step(self, X, y):
        """Perform one gradient descent step."""
        gradient = self.compute_gradient(X, y)
        self.weights -= self.lr * gradient
        return gradient
    
    def get_weights(self):
        return self.weights.copy()
    
    def set_weights(self, weights):
        self.weights = weights.copy()

# Test the model
test_model = SimpleLogisticRegression(N_FEATURES)
test_proba = test_model.predict_proba(X_test_global[:5])
print(f"Model initialized. Test predictions: {test_proba.round(3)}")

---

## 2. Aggregation Methods

We implement four aggregation methods with varying Byzantine robustness.

In [ ]:
def fedavg(updates: List[np.ndarray], weights: List[float] = None) -> np.ndarray:
    """
    Federated Averaging: weighted average of all updates.
    No Byzantine robustness - vulnerable to any malicious participant.
    """
    if weights is None:
        weights = [1.0 / len(updates)] * len(updates)
    
    aggregated = np.zeros_like(updates[0])
    for update, weight in zip(updates, weights):
        aggregated += weight * update
    
    return aggregated

def coordinate_median(updates: List[np.ndarray]) -> np.ndarray:
    """
    Coordinate-wise median: take median of each parameter independently.
    Tolerates up to 50% Byzantine participants.
    """
    stacked = np.stack(updates, axis=0)
    return np.median(stacked, axis=0)

def trimmed_mean(updates: List[np.ndarray], trim_ratio: float = 0.2) -> np.ndarray:
    """
    Trimmed mean: remove extreme values, average the rest.
    trim_ratio: proportion to remove from each end.
    """
    stacked = np.stack(updates, axis=0)
    n_participants = len(updates)
    n_trim = int(n_participants * trim_ratio)
    
    if n_trim == 0:
        return np.mean(stacked, axis=0)
    
    # Sort and trim for each coordinate
    sorted_updates = np.sort(stacked, axis=0)
    trimmed = sorted_updates[n_trim:-n_trim]
    
    return np.mean(trimmed, axis=0)

def krum(updates: List[np.ndarray], n_byzantine: int = None) -> np.ndarray:
    """
    Krum: select the update closest to others.
    n_byzantine: assumed number of Byzantine participants.
    """
    n = len(updates)
    if n_byzantine is None:
        n_byzantine = int(n * 0.2)  # Default: assume 20% Byzantine
    
    # Compute pairwise distances
    distances = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.linalg.norm(updates[i] - updates[j])
            distances[i, j] = dist
            distances[j, i] = dist
    
    # For each update, sum distances to n - n_byzantine - 2 closest others
    n_closest = n - n_byzantine - 2
    n_closest = max(1, n_closest)
    
    scores = []
    for i in range(n):
        sorted_distances = np.sort(distances[i])
        score = np.sum(sorted_distances[1:n_closest + 1])  # Exclude self (distance 0)
        scores.append(score)
    
    # Select update with minimum score (closest to others)
    best_idx = np.argmin(scores)
    return updates[best_idx]

print("✓ Aggregation methods implemented")

---

## 3. Byzantine Attack Implementations

In [ ]:
def random_noise_attack(honest_gradient: np.ndarray, noise_scale: float = 10.0) -> np.ndarray:
    """
    Random noise attack: replace gradient with random noise.
    Simple attack that disrupts convergence.
    """
    return np.random.randn(*honest_gradient.shape) * noise_scale

def sign_flip_attack(honest_gradient: np.ndarray, scale: float = 3.0) -> np.ndarray:
    """
    Sign flip attack: negate the gradient and scale it.
    Pushes model in wrong direction.
    """
    return -scale * honest_gradient

def targeted_attack(honest_gradient: np.ndarray, target_indices: List[int], 
                    perturbation: float = 5.0) -> np.ndarray:
    """
    Targeted attack: modify specific parameters.
    More subtle - may affect specific predictions.
    """
    malicious = honest_gradient.copy()
    for idx in target_indices:
        if idx < len(malicious):
            malicious[idx] += perturbation * np.sign(np.random.randn())
    return malicious

def label_flip_attack(model: SimpleLogisticRegression, X: np.ndarray, 
                      y: np.ndarray, flip_rate: float = 0.3) -> np.ndarray:
    """
    Label flip attack: train on corrupted labels.
    Byzantine participant flips some labels before training.
    """
    y_corrupted = y.copy()
    n_flip = int(len(y) * flip_rate)
    flip_idx = np.random.choice(len(y), n_flip, replace=False)
    y_corrupted[flip_idx] = 1 - y_corrupted[flip_idx]
    
    return model.compute_gradient(X, y_corrupted)

print("✓ Attack methods implemented")

---

## 4. Federated Learning Simulation

In [ ]:
class FederatedLearningSimulator:
    """
    Simulate federated learning with Byzantine participants.
    """
    
    def __init__(self, hospital_data: List[Dict], n_features: int,
                 byzantine_ids: List[int] = None, attack_type: str = 'random_noise'):
        """
        Parameters:
        -----------
        hospital_data : list of dicts with 'X', 'y' for each hospital
        n_features : number of features
        byzantine_ids : list of hospital indices that are Byzantine
        attack_type : 'random_noise', 'sign_flip', 'targeted', 'label_flip'
        """
        self.hospital_data = hospital_data
        self.n_hospitals = len(hospital_data)
        self.n_features = n_features
        self.byzantine_ids = byzantine_ids or []
        self.attack_type = attack_type
        
        # Initialize global model
        self.global_model = SimpleLogisticRegression(n_features)
        
        # Track history
        self.history = []
    
    def local_train(self, hospital_id: int, global_weights: np.ndarray,
                    local_epochs: int = 1) -> np.ndarray:
        """
        Perform local training at one hospital.
        Returns gradient update.
        """
        X = self.hospital_data[hospital_id]['X']
        y = self.hospital_data[hospital_id]['y']
        
        # Create local model with global weights
        local_model = SimpleLogisticRegression(self.n_features)
        local_model.set_weights(global_weights)
        
        # Compute honest gradient
        honest_gradient = local_model.compute_gradient(X, y)
        
        # Apply attack if Byzantine
        if hospital_id in self.byzantine_ids:
            if self.attack_type == 'random_noise':
                return random_noise_attack(honest_gradient)
            elif self.attack_type == 'sign_flip':
                return sign_flip_attack(honest_gradient)
            elif self.attack_type == 'targeted':
                return targeted_attack(honest_gradient, [0, 1, 2])
            elif self.attack_type == 'label_flip':
                return label_flip_attack(local_model, X, y)
        
        return honest_gradient
    
    def train_round(self, aggregation_fn: Callable, **agg_kwargs) -> Dict:
        """
        Execute one round of federated learning.
        """
        global_weights = self.global_model.get_weights()
        
        # Collect updates from all hospitals
        updates = []
        for h_id in range(self.n_hospitals):
            gradient = self.local_train(h_id, global_weights)
            updates.append(gradient)
        
        # Aggregate updates
        aggregated_gradient = aggregation_fn(updates, **agg_kwargs)
        
        # Update global model
        new_weights = global_weights - 0.1 * aggregated_gradient
        self.global_model.set_weights(new_weights)
        
        return {'updates': updates, 'aggregated': aggregated_gradient}
    
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict:
        """
        Evaluate global model.
        """
        y_pred_proba = self.global_model.predict_proba(X_test)
        y_pred = self.global_model.predict(X_test)
        
        return {
            'auroc': roc_auc_score(y_test, y_pred_proba),
            'accuracy': accuracy_score(y_test, y_pred)
        }
    
    def train(self, n_rounds: int, aggregation_fn: Callable,
              X_test: np.ndarray, y_test: np.ndarray, **agg_kwargs) -> List[Dict]:
        """
        Train for multiple rounds and track metrics.
        """
        # Reset model
        self.global_model = SimpleLogisticRegression(self.n_features)
        history = []
        
        for round_num in range(n_rounds):
            self.train_round(aggregation_fn, **agg_kwargs)
            metrics = self.evaluate(X_test, y_test)
            metrics['round'] = round_num
            history.append(metrics)
        
        return history

print("✓ Federated learning simulator implemented")

---

## 5. Comparing Aggregation Methods Under Attack

In [ ]:
def run_experiment(byzantine_ids: List[int], attack_type: str, n_rounds: int = 50):
    """
    Run federated learning experiment with different aggregation methods.
    """
    aggregators = {
        'FedAvg': (fedavg, {}),
        'Median': (coordinate_median, {}),
        'Trimmed Mean (20%)': (trimmed_mean, {'trim_ratio': 0.2}),
        'Krum': (krum, {'n_byzantine': len(byzantine_ids)})
    }
    
    results = {}
    
    for name, (agg_fn, kwargs) in aggregators.items():
        sim = FederatedLearningSimulator(
            hospital_data=hospital_data,
            n_features=N_FEATURES,
            byzantine_ids=byzantine_ids,
            attack_type=attack_type
        )
        
        history = sim.train(
            n_rounds=n_rounds,
            aggregation_fn=agg_fn,
            X_test=X_test_global,
            y_test=y_test_global,
            **kwargs
        )
        
        results[name] = history
    
    return results

# Run with 2 Byzantine participants (20%) using random noise attack
byzantine_hospitals = [0, 1]  # Hospitals 0 and 1 are compromised
print(f"Experiment: {len(byzantine_hospitals)} Byzantine participants out of {N_HOSPITALS}")
print(f"Attack type: Random Noise")
print(f"Training for 50 rounds...\n")

results_noise = run_experiment(byzantine_hospitals, 'random_noise', n_rounds=50)

In [ ]:
# Visualize results
def plot_comparison(results: Dict, title: str, metric: str = 'auroc'):
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors_list = [COLORS['fedavg'], COLORS['median'], COLORS['trimmed'], COLORS['krum']]
    
    for (name, history), color in zip(results.items(), colors_list):
        rounds = [h['round'] for h in history]
        values = [h[metric] for h in history]
        ax.plot(rounds, values, label=name, linewidth=2, color=color)
    
    ax.set_xlabel('Training Round', fontsize=12)
    ax.set_ylabel(f'{metric.upper()}', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.4, 1.0)
    
    return fig, ax

fig, ax = plot_comparison(
    results_noise,
    f'Aggregation Methods Under Random Noise Attack\n({len(byzantine_hospitals)}/{N_HOSPITALS} Byzantine Participants)'
)
plt.tight_layout()
plt.show()

# Print final performance
print("\nFinal AUROC after 50 rounds:")
for name, history in results_noise.items():
    final_auroc = history[-1]['auroc']
    print(f"  {name}: {final_auroc:.3f}")

In [ ]:
# Test with sign flip attack (more aggressive)
print("\nRunning with Sign Flip Attack...")
results_sign_flip = run_experiment(byzantine_hospitals, 'sign_flip', n_rounds=50)

fig, ax = plot_comparison(
    results_sign_flip,
    f'Aggregation Methods Under Sign Flip Attack\n({len(byzantine_hospitals)}/{N_HOSPITALS} Byzantine Participants)'
)
plt.tight_layout()
plt.show()

print("\nFinal AUROC after 50 rounds:")
for name, history in results_sign_flip.items():
    final_auroc = history[-1]['auroc']
    print(f"  {name}: {final_auroc:.3f}")

---

## 6. Varying the Number of Byzantine Participants

In [ ]:
def evaluate_byzantine_tolerance(attack_type: str, n_rounds: int = 30):
    """
    Evaluate aggregation methods with varying numbers of Byzantine participants.
    """
    byzantine_counts = [0, 1, 2, 3, 4]  # Out of 10 hospitals
    
    tolerance_results = {}
    
    for n_byz in byzantine_counts:
        byzantine_ids = list(range(n_byz))
        results = run_experiment(byzantine_ids, attack_type, n_rounds)
        
        for name, history in results.items():
            if name not in tolerance_results:
                tolerance_results[name] = []
            tolerance_results[name].append({
                'n_byzantine': n_byz,
                'pct_byzantine': n_byz / N_HOSPITALS * 100,
                'final_auroc': history[-1]['auroc']
            })
    
    return tolerance_results

print("Evaluating Byzantine tolerance across aggregation methods...")
print("Testing with 0, 1, 2, 3, 4 Byzantine participants (out of 10)\n")

tolerance_results = evaluate_byzantine_tolerance('sign_flip', n_rounds=30)

In [ ]:
# Visualize Byzantine tolerance
fig, ax = plt.subplots(figsize=(12, 6))

colors_list = [COLORS['fedavg'], COLORS['median'], COLORS['trimmed'], COLORS['krum']]

for (name, results), color in zip(tolerance_results.items(), colors_list):
    pct_byzantine = [r['pct_byzantine'] for r in results]
    aurocs = [r['final_auroc'] for r in results]
    ax.plot(pct_byzantine, aurocs, 'o-', label=name, linewidth=2, 
            markersize=10, color=color)

ax.axhline(y=0.5, color='gray', linestyle='--', label='Random Baseline')
ax.set_xlabel('Percentage of Byzantine Participants (%)', fontsize=12)
ax.set_ylabel('Final AUROC', fontsize=12)
ax.set_title('Byzantine Fault Tolerance: Sign Flip Attack', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 42)
ax.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.show()

# Create summary table
summary_data = []
for name, results in tolerance_results.items():
    row = {'Method': name}
    for r in results:
        row[f"{int(r['pct_byzantine'])}% Byz"] = f"{r['final_auroc']:.3f}"
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
print("\nByzantine Tolerance Summary (Final AUROC):")
print(df_summary.to_string(index=False))

---

## 7. Computational Cost Analysis

Byzantine-robust methods have different computational requirements.

In [ ]:
import time

def measure_aggregation_time(n_participants: int, n_parameters: int, n_trials: int = 100):
    """
    Measure aggregation time for different methods.
    """
    # Generate random updates
    updates = [np.random.randn(n_parameters) for _ in range(n_participants)]
    
    methods = {
        'FedAvg': (fedavg, {}),
        'Median': (coordinate_median, {}),
        'Trimmed Mean': (trimmed_mean, {'trim_ratio': 0.2}),
        'Krum': (krum, {'n_byzantine': int(n_participants * 0.2)})
    }
    
    times = {}
    
    for name, (fn, kwargs) in methods.items():
        start = time.time()
        for _ in range(n_trials):
            fn(updates, **kwargs)
        elapsed = (time.time() - start) / n_trials * 1000  # ms
        times[name] = elapsed
    
    return times

# Test with different scales
scales = [
    (10, 100, 'Small (10 hospitals, 100 params)'),
    (50, 1000, 'Medium (50 hospitals, 1K params)'),
    (100, 10000, 'Large (100 hospitals, 10K params)')
]

timing_results = []

print("Measuring aggregation times...\n")
for n_part, n_param, label in scales:
    times = measure_aggregation_time(n_part, n_param, n_trials=50)
    for method, time_ms in times.items():
        timing_results.append({
            'Scale': label,
            'Method': method,
            'Time (ms)': time_ms
        })

df_timing = pd.DataFrame(timing_results)
pivot_timing = df_timing.pivot(index='Method', columns='Scale', values='Time (ms)')
print("Aggregation Time (milliseconds):")
print(pivot_timing.round(2).to_string())

In [ ]:
# Visualize computational costs
fig, ax = plt.subplots(figsize=(10, 6))

methods_order = ['FedAvg', 'Median', 'Trimmed Mean', 'Krum']
scale_labels = [s[2] for s in scales]

x = np.arange(len(methods_order))
width = 0.25

for i, scale_label in enumerate(scale_labels):
    times = [pivot_timing.loc[m, scale_label] for m in methods_order]
    ax.bar(x + i * width, times, width, label=scale_label)

ax.set_xlabel('Aggregation Method', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Computational Cost of Aggregation Methods', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(methods_order)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Krum has O(n²) complexity due to pairwise distance computation,")
print("   making it expensive for large consortia. Trimmed mean provides a good")
print("   balance between robustness and computational efficiency.")

---

## 8. Practical Recommendations for Healthcare FL

In [ ]:
# Summary comparison table
comparison_data = [
    {
        'Method': 'FedAvg',
        'Byzantine Tolerance': '0%',
        'Computational Cost': 'O(n)',
        'Convergence Speed': 'Fast',
        'Recommendation': 'Only use with fully trusted participants'
    },
    {
        'Method': 'Coordinate Median',
        'Byzantine Tolerance': '<50%',
        'Computational Cost': 'O(n log n)',
        'Convergence Speed': 'Medium',
        'Recommendation': 'Good default for moderate-risk settings'
    },
    {
        'Method': 'Trimmed Mean',
        'Byzantine Tolerance': '10-20%',
        'Computational Cost': 'O(n log n)',
        'Convergence Speed': 'Medium-Fast',
        'Recommendation': 'Best balance of robustness and efficiency'
    },
    {
        'Method': 'Krum',
        'Byzantine Tolerance': '<50%',
        'Computational Cost': 'O(n²)',
        'Convergence Speed': 'Slow',
        'Recommendation': 'Use for small, high-security consortia'
    }
]

df_comparison = pd.DataFrame(comparison_data)
print("Byzantine-Robust Aggregation Method Comparison")
print("=" * 90)
print(df_comparison.to_string(index=False))

### Clinical Example: Designing a Sepsis Prediction Consortium

Consider a consortium of 10 hospitals developing a federated sepsis model:

**Threat Assessment:**
- Hospitals are generally trusted but IT systems may be compromised
- Software bugs could cause unintentionally bad updates
- Expect 1-2 problematic participants at any time (10-20%)

**Recommended Configuration:**
1. Use **Trimmed Mean with 20% trimming** as the primary aggregation
2. Monitor update magnitudes and flag outliers for investigation
3. Implement **reputation tracking** to identify consistently problematic participants
4. Require **minimum participation threshold** (e.g., 7/10 hospitals) for each round
5. Perform **independent validation** at each site before deploying global updates

---

## 9. Summary

### Key Takeaways

1. **FedAvg provides no Byzantine protection**: A single malicious participant can arbitrarily influence the global model.

2. **Coordinate-wise median is simple and effective**: Tolerates up to 50% Byzantine participants with minimal computational overhead.

3. **Trimmed mean provides tunable robustness**: The trim ratio can be adjusted based on expected threat level.

4. **Krum is most selective but expensive**: Best for small consortia where security is paramount.

5. **Defense effectiveness depends on attack type**: Some attacks (e.g., subtle targeted attacks) are harder to detect than random noise.

6. **Multiple defenses should be combined**: Byzantine-robust aggregation + monitoring + reputation systems provides defense in depth.

In [ ]:
print("\n" + "="*70)
print("NOTEBOOK COMPLETE: Byzantine-Robust Federated Learning")
print("="*70)
print("""
✅ Implemented federated learning simulation with multiple hospitals
✅ Created Byzantine attack scenarios (random noise, sign flip, targeted)
✅ Compared aggregation methods: FedAvg, Median, Trimmed Mean, Krum
✅ Evaluated Byzantine tolerance at different attack intensities
✅ Analyzed computational costs for each method

For healthcare federated learning, use Byzantine-robust aggregation
by default - the computational overhead is small compared to the
security benefits.
""")

---

## Exercises

1. **Adaptive attacks**: Implement an attacker that observes the aggregation method and adapts its strategy. How effective are defenses against adaptive attackers?

2. **Reputation system**: Add a reputation mechanism that tracks participant behavior over rounds. Down-weight or exclude participants with poor reputation.

3. **Secure aggregation**: Research and discuss how secure aggregation (where the server cannot see individual updates) would affect Byzantine detection.

4. **Non-IID data**: Modify the data generation to create highly heterogeneous (non-IID) data across hospitals. How does this affect the distinction between Byzantine and honest updates?

5. **Real consortium simulation**: Design a complete federated learning protocol for a real healthcare use case, including aggregation method selection, monitoring, and incident response.

---

*This notebook accompanies Chapter 2 of AI in Healthcare Volume 2.*